In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
ROOT_DIR = Path("..").resolve() 
DATA_DIR = ROOT_DIR / "data"

TRAIN_PATH = DATA_DIR / "atm_transactions_train.csv"
TEST_PATH = DATA_DIR / "atm_transactions_test.csv"

print("ROOT_DIR:", ROOT_DIR)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("TEST_PATH exists:", TEST_PATH.exists())




ROOT_DIR: /Users/khaledalrashidi/Desktop/Projects/gulf_bank_datathon
TRAIN_PATH exists: True
TEST_PATH exists: True


In [3]:
# ------------------------------------------------------------------
# Data loading & cleaning
# ------------------------------------------------------------------
def load_and_clean_train(path: Path) -> pd.DataFrame:
    """
    Load the training CSV and return a clean DataFrame with columns:
    - dt (datetime)
    - atm_id
    - withdrawn_kwd      (total withdrawal amount in KWD)
    - withdraw_count     (number of withdrawal transactions)
    """
    df = pd.read_csv(path)

    # 1) Rename raw columns to standard internal names
    rename_map = {
        "dt": "dt",
        "atm_id": "atm_id",
        "total_withdrawn_amount_kwd": "withdrawn_kwd",
        "total_withdraw_txn_count": "withdraw_count",
    }
    df = df.rename(columns=rename_map)

    # 2) Sanity check
    required = ["dt", "atm_id", "withdrawn_kwd", "withdraw_count"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing expected columns after rename: {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )

    # 3) Standard cleaning
    df["dt"] = pd.to_datetime(df["dt"])

    # Keep only needed columns and aggregate in case of duplicates
    df = (
        df[required]
        .drop_duplicates()
        .groupby(["atm_id", "dt"], as_index=False)[["withdrawn_kwd", "withdraw_count"]]
        .sum()
    )

    return df


In [4]:
# ------------------------------------------------------------------
# Baseline models
# ------------------------------------------------------------------
from typing import Optional


def make_baseline_predictions_const(
    train_df: pd.DataFrame,
    test_like_df: pd.DataFrame,
    kind: str,
    window: Optional[int] = None,
) -> pd.DataFrame:
    """
    Create baseline predictions that are constant per atm_id across all rows
    in test_like_df.

    Parameters
    ----------
    train_df : DataFrame
        Must contain columns: dt, atm_id, withdrawn_kwd, withdraw_count
    test_like_df : DataFrame
        Must contain columns: dt, atm_id
    kind : str
        "last" or "moving_average"
    window : int, optional
        For moving_average, size of the window in days
    """
    train_sorted = train_df.sort_values("dt")
    gb = train_sorted.groupby("atm_id", group_keys=False)

    if kind == "last":
        # Take last observed value per ATM
        stats = gb[["withdrawn_kwd", "withdraw_count"]].last()

    elif kind == "moving_average":
        if window is None:
            raise ValueError("window must be provided for moving_average")

        def agg_fn(g: pd.DataFrame) -> pd.Series:
            tail = g.sort_values("dt").tail(window)
            return tail[["withdrawn_kwd", "withdraw_count"]].mean()

        stats = gb.apply(agg_fn)

    else:
        raise ValueError(f"Unknown baseline kind: {kind}")

    # Rename to prediction column names
    stats = stats.rename(
        columns={
            "withdrawn_kwd": "predicted_withdrawn_kwd",
            "withdraw_count": "predicted_withdraw_count",
        }
    )

    # Merge per-ATM stats onto all (dt, atm_id) rows in test_like_df
    preds = test_like_df[["dt", "atm_id"]].merge(
        stats, left_on="atm_id", right_index=True, how="left"
    )

    # Fallback: if any ATM in test_like_df is missing from train_df (unlikely)
    if preds["predicted_withdrawn_kwd"].isna().any():
        global_means = train_df[["withdrawn_kwd", "withdraw_count"]].mean()

        preds["predicted_withdrawn_kwd"] = preds["predicted_withdrawn_kwd"].fillna(
            global_means["withdrawn_kwd"]
        )
        preds["predicted_withdraw_count"] = preds["predicted_withdraw_count"].fillna(
            global_means["withdraw_count"]
        )

    return preds


In [5]:
# ------------------------------------------------------------------
# Metric
# ------------------------------------------------------------------
def rmse(y_true, y_pred) -> float:
    """
    Compute RMSE without relying on sklearn version details.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


In [6]:
# ------------------------------------------------------------------
# Load data and inspect
# ------------------------------------------------------------------
df = load_and_clean_train(TRAIN_PATH)
print("\nCleaned train data:")
display(df.head())

print("Date range:", df["dt"].min().date(), "->", df["dt"].max().date())
print("Number of ATMs:", df["atm_id"].nunique())


# ------------------------------------------------------------------
# Create time-based validation split (last 14 days)
# ------------------------------------------------------------------
max_dt = df["dt"].max()
val_start = max_dt - pd.Timedelta(days=13)  # inclusive last 14 days

df_train_calib = df[df["dt"] < val_start].copy()
df_val = df[df["dt"] >= val_start].copy()

print("\nTraining calibration period:",
      df_train_calib["dt"].min().date(), "->", df_train_calib["dt"].max().date())
print("Validation period:",
      df_val["dt"].min().date(), "->", df_val["dt"].max().date())
print("Validation ATM-days:", len(df_val))

val_keys = df_val[["dt", "atm_id"]].drop_duplicates()


# ------------------------------------------------------------------
# Evaluate baselines
# ------------------------------------------------------------------
baselines = {
    "naive_last": dict(kind="last", window=None),
    "ma_7": dict(kind="moving_average", window=7),
    "ma_14": dict(kind="moving_average", window=14),
    "ma_28": dict(kind="moving_average", window=28),
}

results = []

for name, cfg in baselines.items():
    print(f"\nEvaluating baseline: {name}")
    preds = make_baseline_predictions_const(
        df_train_calib, val_keys, kind=cfg["kind"], window=cfg["window"]
    )
    merged = df_val.merge(preds, on=["dt", "atm_id"], how="left")

    rmse_kwd = rmse(merged["withdrawn_kwd"], merged["predicted_withdrawn_kwd"])
    rmse_cnt = rmse(merged["withdraw_count"], merged["predicted_withdraw_count"])
    avg_rmse = (rmse_kwd + rmse_cnt) / 2.0

    print(f"  RMSE (withdrawn_kwd):   {rmse_kwd:,.2f}")
    print(f"  RMSE (withdraw_count): {rmse_cnt:,.2f}")
    print(f"  Average RMSE:           {avg_rmse:,.2f}")

    results.append((name, rmse_kwd, rmse_cnt, avg_rmse))

results_df = pd.DataFrame(
    results, columns=["model", "rmse_kwd", "rmse_count", "rmse_avg"]
)

print("\nBaseline comparison:")
display(results_df)


Cleaned train data:


,atm_id,dt,withdrawn_kwd,withdraw_count
0,ATM_0001,2020-07-16,713.23,31
1,ATM_0001,2020-07-17,955.08,33
2,ATM_0001,2020-07-18,974.74,33
3,ATM_0001,2020-07-19,688.74,24
4,ATM_0001,2020-07-20,1360.88,47


Date range: 2020-01-04 -> 2025-10-27
Number of ATMs: 253

Training calibration period: 2020-01-04 -> 2025-10-13
Validation period: 2025-10-14 -> 2025-10-27
Validation ATM-days: 2853

Evaluating baseline: naive_last
  RMSE (withdrawn_kwd):   349.36
  RMSE (withdraw_count): 10.76
  Average RMSE:           180.06

Evaluating baseline: ma_7
  RMSE (withdrawn_kwd):   292.40
  RMSE (withdraw_count): 9.09
  Average RMSE:           150.74

Evaluating baseline: ma_14
  RMSE (withdrawn_kwd):   285.03
  RMSE (withdraw_count): 8.88
  Average RMSE:           146.95

Evaluating baseline: ma_28
  RMSE (withdrawn_kwd):   263.12
  RMSE (withdraw_count): 8.10
  Average RMSE:           135.61

Baseline comparison:


/var/folders/ps/bsyj9bqj7110t7m59qgl73l40000gn/T/ipykernel_36709/2243686644.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats = gb.apply(agg_fn)
/var/folders/ps/bsyj9bqj7110t7m59qgl73l40000gn/T/ipykernel_36709/2243686644.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats = gb.apply(agg_fn)
/var/folders/ps/bsyj9bqj7110t7m59qgl73l40000gn/T/ipykernel_36709/2243686644.py:43: FutureWarning: DataFrameGroup

,model,rmse_kwd,rmse_count,rmse_avg
0,naive_last,349.358310,10.756974,180.057642
1,ma_7,292.397876,9.090421,150.744148
2,ma_14,285.030287,8.879525,146.954906
3,ma_28,263.120524,8.103012,135.611768


In [7]:
# Use all available training data
df_full = load_and_clean_train(TRAIN_PATH)

# Load test keys
df_test = pd.read_csv(TEST_PATH)
df_test["dt"] = pd.to_datetime(df_test["dt"])
test_keys = df_test[["dt", "atm_id"]].drop_duplicates()

# Use the best baseline: 28-day moving average
preds_test = make_baseline_predictions_const(
    df_full,
    test_keys,
    kind="moving_average",
    window=28,
)

predictions = test_keys.merge(
    preds_test[["dt", "atm_id", "predicted_withdrawn_kwd", "predicted_withdraw_count"]],
    on=["dt", "atm_id"],
    how="left",
)

# Sanity check
print("rows test:", len(test_keys), "rows preds:", len(predictions))

# Save file in final required name
predictions.to_csv("predictions.csv", index=False)
predictions.head()

rows test: 2886 rows preds: 2886


/var/folders/ps/bsyj9bqj7110t7m59qgl73l40000gn/T/ipykernel_36709/2243686644.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats = gb.apply(agg_fn)


,dt,atm_id,predicted_withdrawn_kwd,predicted_withdraw_count
0,2025-10-28,ATM_0004,1228.320000,42.142857
1,2025-10-28,ATM_0005,1292.950000,45.892857
2,2025-10-28,ATM_0006,894.881786,30.285714
3,2025-10-28,ATM_0007,1038.813214,36.535714
4,2025-10-28,ATM_0009,1101.287857,43.357143
